# Motivating Example: 2-TX Duke Scene

**Scene:** Duke — two transmitters, Zone 1 (partial LOS) and Zone 2 (NLOS-heavy)  
**Frequency:** 3.5 GHz

Four baselines, each revealing a distinct weakness that motivates the proposed method:

| # | Baseline | Category | Weakness Exposed |
|---|----------|----------|------------------|
| 1 | `naive_edge_center` | Geometric heuristic | Empirical ≠ ray-traced coverage |
| 2 | `empirical_pso` | Gradient-free + empirical | Channel model blind to blockage |
| 3 | `radiomap_gradient` | First-order + RadioMap | Near-zero gradients (binning approx.) |
| 4 | `dense_pathsolver_gradient` | First-order + PathSolver dense | Exact grads but O(zone²) per step |

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
SCENE_XML_PATH = "../scene/scenes/Duke/scene.xml"
CARRIER_HZ     = 3.5e9

# Iteration / sample budget (keep modest for the motivating example)
MOTIV_NUM_ITER       = 50
MOTIV_SAMPLES_PER_TX = int(1e7)

# ── TRANSMITTER 1 (same building as single-TX motivating example) ─────────────
TX1_NAME        = "gnb1"
TX1_BUILDING_ID = 33
TX1_HEIGHT_M    = 10.0

# ── TRANSMITTER 2 (NLOS-heavy zone on opposite side) ─────────────────────────
# NOTE: Confirm TX2_BUILDING_ID via the zone visualization cell below.
# Pick a building whose rooftop has sightlines partially blocked by surrounding
# buildings toward Zone 2 — makes the NLOS scenario visually clear.
TX2_NAME        = "gnb2"
TX2_BUILDING_ID = 20   # ← SET THIS after running the scene visualization
TX2_HEIGHT_M    = 10.0

# ── ZONE 1 (partial LOS — northwest quadrant) ─────────────────────────────────
ZONE1_CENTER   = [-200.0, 300.0]
ZONE1_WIDTH_M  = 200.0
ZONE1_HEIGHT_M = 200.0

# ── ZONE 2 (NLOS-heavy — southwest quadrant) ──────────────────────────────────
ZONE2_CENTER   = [-100.0, -100.0]
ZONE2_WIDTH_M  = 150.0
ZONE2_HEIGHT_M = 150.0

# ── RADIO MAP ─────────────────────────────────────────────────────────────────
MAP_CONFIG = {
    'center':        [0.0, 0.0, 0.0],
    'size':          [1400, 1400],
    'cell_size':     (0.5, 0.5),
    'ground_height': 0.0,
}

# ── EXPERIMENT ────────────────────────────────────────────────────────────────
NOISE_POWER = 1e-10
JITTER_SEED = 42
JITTER_MAG  = 1e-4

In [2]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import mitsuba as mi

try:
    import sionna.rt
except ImportError:
    os.system("pip install sionna-rt")
    import sionna.rt

from sionna.rt import load_scene, AntennaArray
from sionna.rt.antenna_pattern import antenna_pattern_registry

from scene_parser import extract_building_info
from tx_placement import TxPlacement
from boresight_pathsolver import create_zone_mask
from angle_utils import compute_initial_angles_from_position, azimuth_elevation_to_yaw_pitch
from multi_tx_optimizer import TxConfig
from experiment_runner import (
    compare_all_results, plot_cdf, plot_metric_bars, plot_zone_overview,
)
from motivating_baselines import (
    naive_edge_center_baseline_multi_tx,
    empirical_pso_baseline_multi_tx,
    radiomap_gradient_baseline_multi_tx,
    dense_pathsolver_gradient_baseline_multi_tx,
)

scene = load_scene(SCENE_XML_PATH)
scene.frequency = CARRIER_HZ

single_el = np.array([[0.0, 0.0, 0.0]])
scene.tx_array = AntennaArray(
    antenna_pattern=antenna_pattern_registry.get("tr38901")(polarization="V"),
    normalized_positions=single_el.T,
)
scene.rx_array = AntennaArray(
    antenna_pattern=antenna_pattern_registry.get("iso")(polarization="V"),
    normalized_positions=single_el.T,
)
for rm in scene.radio_materials.values():
    rm.scattering_coefficient = 0.4

building_info = extract_building_info(SCENE_XML_PATH, verbose=False)
print(f"Scene loaded  |  {CARRIER_HZ/1e9:.1f} GHz  |  {len(building_info)} buildings")

2026-04-16 00:22:46 WARN  [HDRFilm] Monochrome mode enabled, setting film output pixel format to 'luminance' (was rgb).
Scene loaded  |  3.5 GHz  |  154 buildings


In [3]:
# ── ZONE MASKS ────────────────────────────────────────────────────────────────
zone_params1 = {'center': ZONE1_CENTER, 'width': ZONE1_WIDTH_M, 'height': ZONE1_HEIGHT_M}
zone_mask1, look_at1, zone_stats1 = create_zone_mask(
    map_config=MAP_CONFIG, zone_type='box', zone_params=zone_params1,
    target_height=1.5, scene_xml_path=SCENE_XML_PATH, exclude_buildings=True,
)

zone_params2 = {'center': ZONE2_CENTER, 'width': ZONE2_WIDTH_M, 'height': ZONE2_HEIGHT_M}
zone_mask2, look_at2, zone_stats2 = create_zone_mask(
    map_config=MAP_CONFIG, zone_type='box', zone_params=zone_params2,
    target_height=1.5, scene_xml_path=SCENE_XML_PATH, exclude_buildings=True,
)

print(f"Zone 1 (NW): {zone_stats1['num_cells']:6d} cells  centroid={[round(v,1) for v in zone_stats1['centroid_xy']]}")
print(f"Zone 2 (SW): {zone_stats2['num_cells']:6d} cells  centroid={[round(v,1) for v in zone_stats2['centroid_xy']]}")

Zone 1 (NW): 106772 cells  centroid=[-201.5, 312.8]
Zone 2 (SW):  59707 cells  centroid=[-99.5, -93.2]


In [4]:
# ── TX PLACEMENTS ─────────────────────────────────────────────────────────────
tx_placer1 = TxPlacement(scene, TX1_NAME, SCENE_XML_PATH, TX1_BUILDING_ID, offset=TX1_HEIGHT_M)
tx_placer1.set_rooftop_center()
tx1     = scene.get(TX1_NAME)
tx1_pos = tx1.position.numpy().flatten().tolist()
print(f"TX1 on building {TX1_BUILDING_ID} at ({tx1_pos[0]:.2f}, {tx1_pos[1]:.2f}, {tx1_pos[2]:.2f})")

assert TX2_BUILDING_ID is not None, "Set TX2_BUILDING_ID in the config cell first!"
tx_placer2 = TxPlacement(scene, TX2_NAME, SCENE_XML_PATH, TX2_BUILDING_ID, offset=TX2_HEIGHT_M)
tx_placer2.set_rooftop_center()
tx2     = scene.get(TX2_NAME)
tx2_pos = tx2.position.numpy().flatten().tolist()
print(f"TX2 on building {TX2_BUILDING_ID} at ({tx2_pos[0]:.2f}, {tx2_pos[1]:.2f}, {tx2_pos[2]:.2f})")

TX1 on building 33 at (-231.59, 261.81, 31.00)


AssertionError: Set TX2_BUILDING_ID in the config cell first!

In [ ]:
# ── INITIAL JITTER ────────────────────────────────────────────────────────────
rng = np.random.default_rng(JITTER_SEED)

def _jitter_tx(tx_obj, tx_pos, zone_stats, rng):
    base_az, base_el = compute_initial_angles_from_position(tx_pos, zone_stats["look_at_xyz"])
    az  = base_az + float(rng.uniform(-JITTER_MAG, JITTER_MAG))
    el  = base_el + float(rng.uniform(-JITTER_MAG, JITTER_MAG))
    yaw, pitch = azimuth_elevation_to_yaw_pitch(az, el)
    tx_obj.orientation = [float(yaw), float(pitch), 0.0]
    return az, el, yaw, pitch

az1, el1, yaw1, pitch1 = _jitter_tx(tx1, tx1_pos, zone_stats1, rng)
az2, el2, yaw2, pitch2 = _jitter_tx(tx2, tx2_pos, zone_stats2, rng)

# Store initial state for restoration between baselines
_init = {
    TX1_NAME: (tx1_pos[:], yaw1, pitch1),
    TX2_NAME: (tx2_pos[:], yaw2, pitch2),
}

def _restore_both_tx():
    """Reset both TXs to their jittered initial state before each baseline."""
    for name, (pos, yaw, pitch) in _init.items():
        tx = scene.get(name)
        tx.position    = mi.Point3f(float(pos[0]), float(pos[1]), float(pos[2]))
        tx.orientation = mi.Point3f(float(yaw), float(pitch), 0.0)

print(f"TX1  Az={az1:.6f}°  El={el1:.6f}°")
print(f"TX2  Az={az2:.6f}°  El={el2:.6f}°")

In [ ]:
# ── ZONE + TX VISUALIZATION ───────────────────────────────────────────────────
zone_entries = [
    dict(mask=zone_mask1, cmap='Blues', color='steelblue',
         tx_pos=tx1_pos, tx_name=TX1_NAME, building_id=TX1_BUILDING_ID,
         centroid_xy=zone_stats1['centroid_xy'], zone_label='Zone 1 centroid', marker='^'),
    dict(mask=zone_mask2, cmap='Oranges', color='darkorange',
         tx_pos=tx2_pos, tx_name=TX2_NAME, building_id=TX2_BUILDING_ID,
         centroid_xy=zone_stats2['centroid_xy'], zone_label='Zone 2 centroid', marker='s'),
]
plot_zone_overview(
    building_info, MAP_CONFIG, zone_entries,
    title='Duke Motivating Example — Zone 1 (blue, NW) | Zone 2 (orange, SW)',
    figsize=(9, 9),
)

In [ ]:
# ── TX CONFIGS ────────────────────────────────────────────────────────────────
tx_configs_2 = [
    TxConfig(name=TX1_NAME, building_id=TX1_BUILDING_ID,
             zone_params=zone_params1, tx_height_offset=TX1_HEIGHT_M,
             num_sample_points=128),
    TxConfig(name=TX2_NAME, building_id=TX2_BUILDING_ID,
             zone_params=zone_params2, tx_height_offset=TX2_HEIGHT_M,
             num_sample_points=128),
]
motiv_results = {}

## Baseline 1 — Naive Edge-Center
Places each TX on the building edge facing its zone centroid and aims geometrically.  
Runs both a 3GPP UMa empirical evaluation and a Sionna ray-traced evaluation to expose the gap.

In [ ]:
_restore_both_tx()

result_edge = naive_edge_center_baseline_multi_tx(
    scene=scene,
    tx_configs=tx_configs_2,
    map_config=MAP_CONFIG,
    scene_xml_path=SCENE_XML_PATH,
    noise_power=NOISE_POWER,
    frequency_hz=CARRIER_HZ,
    verbose=True,
)
motiv_results["naive_edge_center"] = result_edge

print(f"\nEmpirical loss : {result_edge['joint']['empirical_loss']:.4f}")
print(f"Ray-traced loss: {result_edge['joint']['raytraced_loss']:.4f}")
print(f"Gap (Δ)        : {result_edge['joint']['raytraced_loss'] - result_edge['joint']['empirical_loss']:+.4f}")

## Baseline 2 — Empirical PSO (3GPP UMa)
PSO driven by the UMa channel model — no ray tracing, no geometric scene knowledge.  
Shows that the empirical model cannot distinguish LOS from NLOS blockage.

In [ ]:
_restore_both_tx()

result_emp_pso = empirical_pso_baseline_multi_tx(
    scene=scene,
    tx_configs=tx_configs_2,
    map_config=MAP_CONFIG,
    scene_xml_path=SCENE_XML_PATH,
    noise_power=NOISE_POWER,
    frequency_hz=CARRIER_HZ,
    n_particles=20,
    num_iterations=30,
    seed=42,
    verbose=True,
)
motiv_results["empirical_pso"] = result_emp_pso

## Baseline 3 — RadioMap Gradient (Adam)
First-order optimization via `RadioMapSolver`.  Gradient norms are expected to be near-zero due to solid-angle ray-tube binning — signal at a cell only changes when a ray crosses a bin boundary, producing piecewise-constant gradients.

In [ ]:
_restore_both_tx()

result_rm_grad = radiomap_gradient_baseline_multi_tx(
    scene=scene,
    tx_configs=tx_configs_2,
    map_config=MAP_CONFIG,
    scene_xml_path=SCENE_XML_PATH,
    noise_power=NOISE_POWER,
    frequency_hz=CARRIER_HZ,
    learning_rate=3.0,
    num_iterations=MOTIV_NUM_ITER,
    samples_per_tx=MOTIV_SAMPLES_PER_TX,
    n_empirical_pts=500,
    verbose=True,
)
motiv_results["radiomap_gradient"] = result_rm_grad

print(f"\nGrad norm  — mean: {float(np.mean(result_rm_grad['joint']['grad_norm_history'])):.2e}"
      f"  min: {float(np.min(result_rm_grad['joint']['grad_norm_history'])):.2e}"
      f"  max: {float(np.max(result_rm_grad['joint']['grad_norm_history'])):.2e}")

## Baseline 4 — Dense PathSolver Gradient (Adam)
First-order optimization via PathSolver over a full dense receiver grid.  Gradient magnitudes should be significantly larger than Baseline 3, but per-iteration time scales as O(zone²/spacing²).

In [ ]:
_restore_both_tx()

result_dense = dense_pathsolver_gradient_baseline_multi_tx(
    scene=scene,
    tx_configs=tx_configs_2,
    map_config=MAP_CONFIG,
    scene_xml_path=SCENE_XML_PATH,
    noise_power=NOISE_POWER,
    learning_rate=3.5,
    num_iterations=MOTIV_NUM_ITER,
    grid_spacing_m=4.0,
    verbose=True,
)
motiv_results["dense_pathsolver"] = result_dense

print(f"\nReceivers used : {result_dense['joint']['n_receivers']}")
print(f"Grad norm      — mean: {float(np.mean(result_dense['joint']['grad_norm_history'])):.2e}"
      f"  min: {float(np.min(result_dense['joint']['grad_norm_history'])):.2e}"
      f"  max: {float(np.max(result_dense['joint']['grad_norm_history'])):.2e}")
print(f"Mean iter time : {float(np.mean(result_dense['joint']['iter_time_history'])):.1f}s")

## Summary

In [ ]:
# ── GRAD NORM COMPARISON (Baselines 3 vs 4) ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(result_rm_grad['joint']['grad_norm_history'],   label='RadioMap (B3)', color='tab:red')
ax.plot(result_dense['joint']['grad_norm_history'],     label='PathSolver dense (B4)', color='tab:blue')
ax.set_xlabel('Iteration')
ax.set_ylabel('Mean |∇param|')
ax.set_title('Gradient Magnitude per Iteration')
ax.set_yscale('log')
ax.legend()

ax = axes[1]
for label, res, color in [
    ('RadioMap (B3)',       result_rm_grad, 'tab:red'),
    ('PathSolver dense (B4)', result_dense, 'tab:blue'),
]:
    hist = res['joint']['loss_history']
    ax.plot(hist, label=label, color=color)
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title('Loss Curves (Gradient Baselines)')
ax.legend()

plt.suptitle('Duke Motivating Example — Gradient Pathology Analysis', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
print(f"{'Baseline':<30s}  {'Elapsed (s)':>12}  {'Final loss':>12}  {'Iter time (s)':>14}")
print('─' * 75)

for label, res in [
    ('Naive edge-center (B1)',       result_edge),
    ('Empirical PSO UMa (B2)',       result_emp_pso),
    ('RadioMap gradient (B3)',       result_rm_grad),
    ('Dense PathSolver grad (B4)',   result_dense),
]:
    joint   = res['joint']
    elapsed = joint.get('elapsed_time_s', float('nan'))
    loss    = joint['loss_history'][-1] if joint['loss_history'] else float('nan')
    t_iter  = (float(np.mean(joint['iter_time_history']))
               if joint.get('iter_time_history') else float('nan'))
    print(f"  {label:<28s}  {elapsed:>12.1f}  {loss:>12.4f}  {t_iter:>14.2f}")

print()
print("Empirical vs ray-traced gap (Baseline 1):")
print(f"  Empirical loss : {result_edge['joint']['empirical_loss']:.4f}")
print(f"  Ray-traced loss: {result_edge['joint']['raytraced_loss']:.4f}")
print()
print("Gradient magnitudes (mean |∇|):")
print(f"  RadioMap (B3)  : {float(np.mean(result_rm_grad['joint']['grad_norm_history'])):.2e}")
print(f"  PathSolver (B4): {float(np.mean(result_dense['joint']['grad_norm_history'])):.2e}")